# Define Project Trips

In [1]:
import warnings
warnings.simplefilter("always")

### Read Inputs

In [2]:
## Data Preparation and Summary
import pandas as pd
import numpy as np

# Set directory
data_dir = "../input/"
interim_dir = '../output/interim/'
output_dir = '../output/'

df_full = pd.read_excel(data_dir + "od_20240422_sandagca_weighted_combined_draftfinal.xlsx", sheet_name="OD_RESULTS", engine='openpyxl')

In [3]:
df_weight = pd.read_excel(data_dir + "2023OBS_ReWeighted_2024-05-24.xlsx",
                          sheet_name="Sheet1", engine='openpyxl')
df = pd.merge(df_full, df_weight[['ID', 'REWEIGHTED_LINKED', 'REWEIGHTED_UNLINKED']],
              on='ID', how='left')

In [4]:
df_xfer = pd.read_csv(output_dir + '2023OBS_blue_line_xfer_dataset.csv')

In [5]:
print(df_full.shape)
print(df.shape)
print(df_xfer.shape)

(35260, 201)
(35260, 203)
(4396, 8)


In [6]:
df_xfer = df_xfer.rename(columns={'direction': 'xfer_direction'})
df = df.merge(df_xfer[['ID','xfer_position','xfer_on_name', 'xfer_off_name', 'xfer_on_midcoast', 'xfer_direction']],on='ID',how='left')

In [7]:
df[['AGENCY', 'AGENCY_CODE','ROUTE_NUMBER', 'DIRECTION']] = df['ROUTE_DIRECTION[Code]'].str.split('_', expand=True)

In [8]:
df.shape

(35260, 211)

## Define project trips

In [9]:
midcoast_stns = [
    'UTC Station',
    'Executive Drive Station',
    'UCSD Health La Jolla Station',
    'UCSD Central Campus Station',
    'VA Medical Center Station',
    'Nobel Drive Station',
    'Balboa Avenue Station',
    'Clairemont Drive Station',
    'Tecolote Road Station'
]

In [10]:
# new project trip, also filter out record that missing stop_on or stop_off stations: ~26 removed trips
df['project_trip_update'] = np.where(
    (
        (
            (df['ROUTE_NUMBER'] == 'Blue') &
            df['STOP_ON [ADDR]'].notna() &
            df['STOP_OFF [ADDR]'].notna() &
            (
                df['STOP_ON [ADDR]'].isin(midcoast_stns) |
                df['STOP_OFF [ADDR]'].isin(midcoast_stns)
            )
        ) |
        (df['xfer_on_midcoast'] == 1)
    ),
    'yes',
    'no'
)

In [11]:
df.groupby('project_trip_update')['REWEIGHTED_LINKED'].sum()

project_trip_update
no     201259.215637
yes     24735.736688
Name: REWEIGHTED_LINKED, dtype: float64

QAQC

In [12]:
# make sure no Blue xfer on survey route
df[df['xfer_on_midcoast'] == 1]['ROUTE_NUMBER'].unique()

array(['101', '303', '308', '309', '398', '1', '10', '105', '11', '12',
       '13', '18', '2', '20', '201', '202', '204', '215', '225', '235',
       '237', '27', '28', '3', '30', '31', '35', '4', '41', '43', '44',
       '5', '60', '7', '701', '704', '709', '712', '8', '815', '816',
       '83', '832', '84', '848', '851', '852', '854', '855', '856', '88',
       '9', '901', '905', '906', '907', '916', '921', '923', '929', '932',
       '934', '936', '944', '950', '955', '961', '963', '968', '978',
       '985', '992', 'Green', 'Orange'], dtype=object)

In [13]:
df['project_trip_on_survey'] = np.where(
    (
        (
            (df['ROUTE_NUMBER'] == 'Blue')
        ) & df['STOP_ON [ADDR]'].notna() &
            df['STOP_OFF [ADDR]'].notna() &
        (
            df['STOP_ON [ADDR]'].isin(midcoast_stns) |
            df['STOP_OFF [ADDR]'].isin(midcoast_stns)
        )
    ),
    'yes',
    'no'
)

df.groupby('project_trip_on_survey')['REWEIGHTED_LINKED'].sum()

project_trip_on_survey
no     203469.782092
yes     22525.170233
Name: REWEIGHTED_LINKED, dtype: float64

In [14]:
df = df.drop(columns=['project_trip_on_survey'])
df.shape

(35260, 212)

## Define stop on and stop off

In [15]:
df_project = df[df['project_trip_update']=='yes'].copy()
df_project.shape

(2755, 212)

In [16]:
df_project['stop_on_project'] = np.where(
    df_project['ROUTE_NUMBER'] == 'Blue',
    df_project['STOP_ON [ADDR]'],
    df_project['xfer_on_name']
)

df_project['stop_off_project'] = np.where(
    df_project['ROUTE_NUMBER'] == 'Blue',
    df_project['STOP_OFF [ADDR]'],
    df_project['xfer_off_name']
)

In [17]:
# QAQC
# another method: replace stop on with xfer_on
def assign_stop_project(row):
    if row.get('project_trip_update') == 'yes':
        stop_on = row.get('xfer_on_name') if row.get('xfer_on_midcoast') == 1 else row.get('STOP_ON [ADDR]')
        stop_off = row.get('xfer_off_name') if row.get('xfer_on_midcoast') == 1 else row.get('STOP_OFF [ADDR]')
    else:
        stop_on = None
        stop_off = None
    return pd.Series([stop_on, stop_off])

df_project[['stop_on_project_round2', 'stop_off_project_round2']] = df_project.apply(assign_stop_project, axis=1, result_type='expand')


In [18]:
df_project['stop_on_match'] = np.where(
    df_project['project_trip_update'] == 'yes',
    df_project['stop_on_project_round2'].fillna('___NULL___') == df_project['stop_on_project'].fillna('___NULL___'),
    None
)
df_project['stop_on_match'].unique()

array([True], dtype=object)

In [19]:
df_project['stop_off_match'] = np.where(
    df_project['project_trip_update'] == 'yes',
    df_project['stop_off_project_round2'].fillna('___NULL___') == df_project['stop_off_project'].fillna('___NULL___'),
    None
)
df_project['stop_off_match'].unique()

array([True], dtype=object)

In [20]:
df_project = df_project.drop(columns=['stop_on_project_round2','stop_off_project_round2','stop_on_match', 'stop_off_match'])

In [21]:
df_project.shape

(2755, 214)

## Define Direction

In [22]:
df_project.columns

Index(['ID', 'DATE_COMPLETED', 'ROUTE_DIRECTION[Code]', 'ROUTE_DIRECTION',
       'ROUTE_DIRECTION [Other]', 'USA_OR_MEXICO[Code]', 'USA_OR_MEXICO',
       'MEXICO_COLONIA [SQ001]', 'MEXICO_COLONIA [SQ002]',
       'PLACE_OF_ENTRY[Code]',
       ...
       'xfer_on_name', 'xfer_off_name', 'xfer_on_midcoast', 'xfer_direction',
       'AGENCY', 'ROUTE_NUMBER', 'DIRECTION', 'project_trip_update',
       'stop_on_project', 'stop_off_project'],
      dtype='object', length=214)

In [23]:
df_project['DIRECTION'] = df_project['DIRECTION'].map({'00': 'NB', '01': 'SB'})

In [24]:
df_project['DIRECTION'].unique()

array(['NB', 'SB'], dtype=object)

In [25]:
df_project['direction_od_project'] = np.where(
    df_project['ROUTE_NUMBER'] == 'Blue',
    df_project['DIRECTION'],
    df_project['xfer_direction']
)


## Define mode of access to project

In [26]:
prev_xfer_cols = ['TRIP_FIRST_ROUTE[Code]','TRIP_SECOND_ROUTE[Code]','TRIP_THIRD_ROUTE[Code]','TRIP_FOURTH_ROUTE[Code]']
next_xfer_cols = ['TRIP_NEXT_ROUTE[Code]','TRIP_AFTER_ROUTE[Code]','TRIP_3RD_ROUTE[Code]','TRIP_LAST4TH_RTE[Code]']

In [27]:
# Define the mapping dictionary for ORIGIN_TRANSPORT
origin_transport_map = {
    'Walk': 'Walk',
    'Was dropped off by someone': 'KNR',
    'Uber, Lyft, etc. (pool or shared)': 'KNR',
    'Taxi': 'KNR',
    'Other shuttle': 'KNR',
    'Uber, Lyft, etc. (private)': 'KNR',
    'Electric vehicle shuttle': 'KNR',
    'Drove alone and parked': 'PNR',
    'Drove or rode with others and parked': 'PNR'
}

df_project['origin_transport_mode'] = df_project['ORIGIN_TRANSPORT'].map(origin_transport_map).fillna('Walk')

# Define the mapping dictionary for DESTIN_TRANSPORT
destin_transport_map = {
    'Walk': 'Walk',
    'Be picked up by someone': 'KNR',
    'Uber, Lyft, etc. (pool or shared)': 'KNR',
    'Taxi': 'KNR',
    'Other shuttle': 'KNR',
    'Uber, Lyft, etc. (private)': 'KNR',
    'Electric vehicle shuttle': 'KNR',
    'Get in a parked vehicle & drive alone': 'PNR',
    'Get in a parked vehicle & drive/ride w/others': 'PNR'
}

df_project['destin_transport_mode'] = df_project['DESTIN_TRANSPORT'].map(destin_transport_map).fillna('Walk')



In [28]:
# NEXT_TRANSFERS>0, then mark the last position
# To define a new column last_position in df_project that finds the last non-NaN column from this list
def find_last_position(row):
    if row.get('NEXT_TRANSFERS[Code]', 0) > 0:
        for col in reversed(next_xfer_cols):
            if pd.notna(row.get(col)):
                return col
    return None

df_project['last_position'] = df_project.apply(find_last_position, axis=1)


### access mode to blue line stns

In [29]:
# if survey route = Blue and 'PREV_TRANSFERS[Code]'=0 
#     then access_project_od = ORIGIN_TRANSPORT
# if survey route = Blue and 'PREV_TRANSFERS[Code]'!=0
#     then access_project_od = 'xfer'
# if survey route != Blue and TRIP_FIRST_ROUTE[Code] = Blue,
#     then access_project_od = ORIGIN_TRANSPORT
# if survey route != Blue and TRIP_FIRST_ROUTE[Code] != Blue,
#     then access_project_od = 'xfer'


# if survey route = Blue and 'NEXT_TRANSFERS[Code]'=0 
#     then egress_project_od = DESTIN_TRANSPORT
# if survey route = Blue and 'NEXT_TRANSFERS[Code]'!=0 
#     then egress_project_od = 'xfer'

# generate a temp column for survey!=Blue: 'last_position' to document the last xfer position for this record
# if survey route != Blue and last_position==xfer_position (meaning Blue is the last xfer):
#     then egress_project_od = DESTIN_TRANSPORT
# if survey route != Blue and last_position!=xfer_position
#     then egress_project_od = 'xfer'

In [30]:
o_condition = [
    (df_project['ROUTE_NUMBER'] == 'Blue') & (df_project['PREV_TRANSFERS[Code]'] == 0),
    (df_project['ROUTE_NUMBER'] == 'Blue') & (df_project['PREV_TRANSFERS[Code]'] > 0),
    (df_project['ROUTE_NUMBER'] != 'Blue') & (df_project['TRIP_FIRST_ROUTE[Code]'] == 'MTS_1_Blue'),
    (df_project['ROUTE_NUMBER'] != 'Blue') & (df_project['TRIP_FIRST_ROUTE[Code]'] != 'MTS_1_Blue')
]

# Define results for each condition
o_choice = [
    df_project['origin_transport_mode'],  
    'Xfer', 
    df_project['origin_transport_mode'], 
    'Xfer'
]

df_project['access_w_xfer_od_project'] = np.select(o_condition, o_choice, default=None)


In [31]:
d_condition = [
    ((df_project['ROUTE_NUMBER'] == 'Blue') & ((df_project['NEXT_TRANSFERS[Code]'] == 0) | (df_project['NEXT_TRANSFERS[Code]'].isna()))),
    (df_project['ROUTE_NUMBER'] == 'Blue') & (df_project['NEXT_TRANSFERS[Code]'] > 0),
    (df_project['ROUTE_NUMBER'] != 'Blue') & (df_project['last_position'] == df_project['xfer_position']),
    (df_project['ROUTE_NUMBER'] != 'Blue') & (df_project['last_position'] != df_project['xfer_position'])
]

d_choices = [
    df_project['destin_transport_mode'],
    'Xfer',
    df_project['destin_transport_mode'],
    'Xfer'
]

df_project['egress_w_xfer_od_project'] = np.select(d_condition, d_choices, default=None)


In [32]:
df_project.to_csv(output_dir + '2023OBS_project_trip_dataset.csv',index=False)